<h2>Main code for search</h2>

In [2]:
# Replace with your Groq API key
import os
os.environ["GROQ_API_KEY"] = "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"

In [6]:
import os
import requests
from bs4 import BeautifulSoup
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 1. Define the Web Scraping Function
def scrape_website(url, limit=3000):
    """
    Fetch website content and return clean text for LLM
    """
    # Adding a simple User-Agent header is good practice for web scraping
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)
    return text[:limit]  # Limit for LLM input context window

# 2. Initialize the LLM
# Note: Ensure your GROQ_API_KEY is set in your environment variables
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

# 3. Create the Prompt and Chain
template = """
You are a helpful healthcare data extraction assistant. Your job is to find specific doctor information from the provided website content based on the user's request.

When the user asks for a type of specialist or a specific doctor, you MUST output the information in the exact format below. 

Format:
- Doctor Name: [Insert Name]
- Location/Branch: [Insert Location or Chamber]
- Contact: [Insert Phone Number or Email]

Important Rules:
1. Only use the information provided in the "Website content" below.
2. If multiple doctors match the criteria, list all of them following the format above.
3. If the specific information (like contact or location) is missing from the text, write "Not provided".
4. If you cannot find any doctor matching the user's request, simply say: "I'm sorry, I couldn't find a doctor matching that description on the website."

Website content:
{website}

User question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | llm

# 4. Execute the Pipeline
if __name__ == "__main__":
    # Scrape the website
    URL = "https://seradoctor.com/service/doctors" 
    print(f"Scraping data from: {URL}...")
    website_text = scrape_website(URL)
    
    # Define the question you want to ask the LLM
    user_question = "Is there any eye doctor nearby "
    
    # Run the LLM chain with the scraped data and the question
    print("Analyzing website content...\n")
    response = chain.invoke({
        "question": user_question,
        "website": website_text
    })
    
    # Print the final answer
    print("--- ASSISTANT RESPONSE ---")
    print(response.content)

Scraping data from: https://seradoctor.com/service/doctors...
Analyzing website content...

--- ASSISTANT RESPONSE ---
I'm sorry, I couldn't find a doctor matching that description on the website.
